# 🏀 FC Bayern Basketball: Multi-Competition Match & Exact Score Prediction
### Model v3: Pure Tactical 1D Time-Decay & Stacked Score Regressors

This notebook trains an end-to-end Machine Learning model to predict both **match winners (win probabilities)** and **exact point scores** for **FC Bayern München Basketball** across three elite competitions:
1. **easyCredit BBL (Basketball Bundesliga)**
2. **Turkish Airlines EuroLeague**
3. **BBL-Pokal (German Cup)**

Unlike football/soccer:
- Basketball games **cannot end in draws** (binary classification outcome).
- Scoring operates on a continuous scale (~65 to 110 points per team).
- Performance is governed by **Pace** (possessions per 40 min), **Offensive Rating (ORtg)**, and **Defensive Rating (DRtg)** per 100 possessions.
- Schedule density and **EuroLeague travel fatigue** (playing in Athens/Madrid on Thursday/Friday and in Germany on Sunday) are primary drivers of performance.

In [ ]:
# 1. Install & Import Core Dependencies
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error
)
import matplotlib.pyplot as plt

print("Environment initialized successfully.")

## 2. Load Multi-Competition Dataset
The dataset spans 6 seasons (2019-2020 through 2024-2025) covering **430 official matches** with exact dates, final scores, pace, and rest days.

In [ ]:
# Path check (supports local run or Kaggle /kaggle/input)
csv_candidates = [
    'basketball_prediction/data/processed/fcbb_multicomp_processed.csv',
    '../input/fcbb-multicomp/fcbb_multicomp_processed.csv',
    'fcbb_multicomp_processed.csv'
]

csv_path = None
for p in csv_candidates:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} matches from {csv_path}")
else:
    print("Note: If running on Kaggle, please attach fcbb_multicomp_processed.csv as input.")
    # Direct load fallback
    df = pd.read_csv('basketball_prediction/data/processed/fcbb_multicomp_processed.csv')

df['date_dt'] = pd.to_datetime(df['date'])
df = df.sort_values('date_dt').reset_index(drop=True)
df.head()

## 3. Feature Engineering: 1D Exponential Time-Decay & Tactical Ratings
We calculate rolling weighted statistics using combined calendar day decay ($e^{-\lambda \cdot \text{days}}$) and game recency decay ($\gamma^{N-1-k}$, with $\gamma = 0.85$).

In [ ]:
def engineer_v3_features(df_in):
    df = df_in.copy()
    comp_weights = {'EuroLeague': 1.25, 'BBL': 1.00, 'BBL-Pokal': 1.05}
    df['comp_tier'] = df['competition'].map(comp_weights).fillna(1.0)
    
    team_histories = {}
    gamma = 0.85
    lambda_days = 0.015
    
    new_cols = {}
    for W in range(1, 11):
        for pfx in ['home', 'away']:
            new_cols[f'{pfx}_pts_decay_{W}'] = []
            new_cols[f'{pfx}_pts_conceded_decay_{W}'] = []
            new_cols[f'{pfx}_pace_decay_{W}'] = []
            new_cols[f'{pfx}_ortg_decay_{W}'] = []
            new_cols[f'{pfx}_drtg_decay_{W}'] = []
            new_cols[f'{pfx}_net_rtg_decay_{W}'] = []
            new_cols[f'{pfx}_win_rate_decay_{W}'] = []
        new_cols[f'net_rtg_diff_{W}'] = []
        new_cols[f'pace_projected_{W}'] = []
        new_cols[f'off_def_mismatch_home_{W}'] = []
        new_cols[f'off_def_mismatch_away_{W}'] = []
        
    for idx, row in df.iterrows():
        ht = row['home_team']
        at = row['away_team']
        dt = row['date_dt']
        
        def get_stats(team, W, cur_date):
            hist = team_histories.get(team, [])
            if not hist:
                return {'pts': 78.0, 'ptsc': 78.0, 'pace': 72.0, 'ortg': 108.0, 'drtg': 108.0, 'net_rtg': 0.0, 'win': 0.50}
            rec = hist[-W:]
            n = len(rec)
            w = np.array([np.exp(-lambda_days * max((cur_date - e['date']).days, 1)) * (gamma ** (n - 1 - k)) * e.get('tier', 1.0) for k, e in enumerate(rec)])
            w = w / np.sum(w) if np.sum(w) > 0 else np.ones(n) / n
            pts = np.sum([e['pts'] for e in rec] * w)
            ptsc = np.sum([e['ptsc'] for e in rec] * w)
            pace = np.sum([e['pace'] for e in rec] * w)
            ortg = np.sum([e['ortg'] for e in rec] * w)
            drtg = np.sum([e['drtg'] for e in rec] * w)
            win = np.sum([1.0 if e['pts'] > e['ptsc'] else 0.0 for e in rec] * w)
            return {'pts': pts, 'ptsc': ptsc, 'pace': pace, 'ortg': ortg, 'drtg': drtg, 'net_rtg': ortg - drtg, 'win': win}
            
        for W in range(1, 11):
            hs = get_stats(ht, W, dt)
            as_ = get_stats(at, W, dt)
            new_cols[f'home_pts_decay_{W}'].append(hs['pts'])
            new_cols[f'home_pts_conceded_decay_{W}'].append(hs['ptsc'])
            new_cols[f'home_pace_decay_{W}'].append(hs['pace'])
            new_cols[f'home_ortg_decay_{W}'].append(hs['ortg'])
            new_cols[f'home_drtg_decay_{W}'].append(hs['drtg'])
            new_cols[f'home_net_rtg_decay_{W}'].append(hs['net_rtg'])
            new_cols[f'home_win_rate_decay_{W}'].append(hs['win'])
            
            new_cols[f'away_pts_decay_{W}'].append(as_['pts'])
            new_cols[f'away_pts_conceded_decay_{W}'].append(as_['ptsc'])
            new_cols[f'away_pace_decay_{W}'].append(as_['pace'])
            new_cols[f'away_ortg_decay_{W}'].append(as_['ortg'])
            new_cols[f'away_drtg_decay_{W}'].append(as_['drtg'])
            new_cols[f'away_net_rtg_decay_{W}'].append(as_['net_rtg'])
            new_cols[f'away_win_rate_decay_{W}'].append(as_['win'])
            
            new_cols[f'net_rtg_diff_{W}'].append(hs['net_rtg'] - as_['net_rtg'])
            new_cols[f'pace_projected_{W}'].append((hs['pace'] + as_['pace']) / 2.0)
            new_cols[f'off_def_mismatch_home_{W}'].append(hs['ortg'] - as_['drtg'])
            new_cols[f'off_def_mismatch_away_{W}'].append(as_['ortg'] - hs['drtg'])
            
        team_histories.setdefault(ht, []).append({'date': dt, 'pts': row['home_score'], 'ptsc': row['away_score'], 'pace': row['pace'], 'ortg': row['home_ortg'], 'drtg': row['away_ortg'], 'tier': row['comp_tier']})
        team_histories.setdefault(at, []).append({'date': dt, 'pts': row['away_score'], 'ptsc': row['home_score'], 'pace': row['pace'], 'ortg': row['away_ortg'], 'drtg': row['home_ortg'], 'tier': row['comp_tier']})
        
    for k, v in new_cols.items():
        df[k] = v
        
    df['home_is_b2b'] = (df['home_rest'] <= 2).astype(int)
    df['away_is_b2b'] = (df['away_rest'] <= 2).astype(int)
    df['b2b_diff'] = df['away_is_b2b'] - df['home_is_b2b']
    df['is_euroleague'] = (df['competition'] == 'EuroLeague').astype(int)
    df['is_bbl'] = (df['competition'] == 'BBL').astype(int)
    df['target_home_win'] = (df['home_score'] > df['away_score']).astype(int)
    return df

df_features = engineer_v3_features(df)
print(f"Feature matrix engineered: {df_features.shape}")

## 4. Horizon Tuning $W \in [1..10]$ & Model Training
We search for the optimal tactical horizon $W^*$, then train the blended winner ensemble and the stacked score regressors.

In [ ]:
# Step 4: Strategy 1 (Multiplicative Pace x Efficiency) & Strategy 2 (Dispersion Expansion)
p_pool = 0.5 * rf_final.predict_proba(X_pool)[:, 1] + 0.5 * hgb_final.predict_proba(X_pool)[:, 1]
X_pool_fused = np.hstack([X_pool, p_pool.reshape(-1, 1), (1 - p_pool).reshape(-1, 1)])
X_test_fused = np.hstack([X_test, p_test.reshape(-1, 1), (1 - p_test).reshape(-1, 1)])

y_pool_pace = df_features.loc[train_pool_mask, 'pace'].values
y_pool_oh = df_features.loc[train_pool_mask, 'home_ortg'].values
y_pool_oa = df_features.loc[train_pool_mask, 'away_ortg'].values

# Component models
reg_pace = HistGradientBoostingRegressor(max_iter=80, max_depth=3, random_state=42).fit(X_pool_fused, y_pool_pace)
reg_oh = HistGradientBoostingRegressor(max_iter=100, max_depth=4, random_state=42).fit(X_pool_fused, y_pool_oh)
reg_oa = HistGradientBoostingRegressor(max_iter=100, max_depth=4, random_state=42).fit(X_pool_fused, y_pool_oa)

pred_pace_test = reg_pace.predict(X_test_fused)
pred_oh_test = reg_oh.predict(X_test_fused)
pred_oa_test = reg_oa.predict(X_test_fused)

raw_pred_hs = pred_pace_test * (pred_oh_test / 100.0)
raw_pred_as = pred_pace_test * (pred_oa_test / 100.0)

# Strategy 2: Calibrated Dispersion Expansion
raw_pool_hs = reg_pace.predict(X_pool_fused) * (reg_oh.predict(X_pool_fused) / 100.0)
raw_pool_as = reg_pace.predict(X_pool_fused) * (reg_oa.predict(X_pool_fused) / 100.0)
mean_h = float(np.mean(y_pool_hs))
mean_a = float(np.mean(y_pool_as))
scale_h = float(np.clip((np.std(y_pool_hs) / np.std(raw_pool_hs)) * 0.85, 1.25, 1.85))
scale_a = float(np.clip((np.std(y_pool_as) / np.std(raw_pool_as)) * 0.85, 1.25, 1.85))

pred_hs = mean_h + scale_h * (raw_pred_hs - mean_h)
pred_as = mean_a + scale_a * (raw_pred_as - mean_a)

for i in range(len(pred_hs)):
    pw = p_test[i]
    if pw >= 0.50 and pred_hs[i] <= pred_as[i]:
        pred_hs[i] = pred_as[i] + max(1.0, (pw - 0.50) * 16.0)
    elif pw < 0.50 and pred_as[i] <= pred_hs[i]:
        pred_as[i] = pred_hs[i] + max(1.0, (0.50 - pw) * 16.0)

print(f'Home Score MAE : {mean_absolute_error(y_test_hs, pred_hs):.2f} pts | Std: {np.std(pred_hs):.2f} (Actual Std: {np.std(y_test_hs):.2f})')
print(f'Away Score MAE : {mean_absolute_error(y_test_as, pred_as):.2f} pts | Std: {np.std(pred_as):.2f} (Actual Std: {np.std(y_test_as):.2f})')
print(f'Score Ranges   : Home [{np.min(pred_hs):.1f} - {np.max(pred_hs):.1f}] | Away [{np.min(pred_as):.1f} - {np.max(pred_as):.1f}]')
print(f'Margin MAE     : {mean_absolute_error(y_test_hs - y_test_as, pred_hs - pred_as):.2f} pts')
print(f'Spread Implied Winner Accuracy: {accuracy_score(y_test_win, (pred_hs > pred_as).astype(int))*100:.2f}%')


## 5. Visualizing Exact Score Predictions vs Actual (Test Season)

In [ ]:
eval_df = df_features.loc[test_mask, ['date', 'competition', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
eval_df['pred_home'] = np.round(pred_hs).astype(int)
eval_df['pred_away'] = np.round(pred_as).astype(int)
eval_df['win_prob'] = np.round(p_test * 100, 1)

plt.figure(figsize=(10, 5))
plt.scatter(eval_df['home_score'], eval_df['pred_home'], color='#dc2626', alpha=0.7, label='Home Points')
plt.scatter(eval_df['away_score'], eval_df['pred_away'], color='#2563eb', alpha=0.7, label='Away Points')
plt.plot([60, 110], [60, 110], 'k--', alpha=0.5, label='Perfect Forecast Line')
plt.xlabel('Actual Points Scored')
plt.ylabel('Model v3 Predicted Points')
plt.title('FC Bayern Basketball: Model v3 Exact Score Calibration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

eval_df.tail(8)